# Article 1 — Experiment 2 full matrix

This notebook reports the preregistered mechanism comparison over **3 datasets × 3 seeds × 6 heterogeneity regimes**.  It separates (i) all-teacher support restriction, (ii) expertise teacher selection, and (iii) their combination.  It uses only completed per-cell `results.csv` artifacts; cells are analysed as paired seed-level contrasts against FedDF.

Run it from the repository root after `RUN_STUDENTS=1 bash experiments/article_1_server_expertise/run_experiment2_full_matrix.sh` has completed.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
paths = sorted(ROOT.glob('../../OUTPUTS/experiments/article1_experiment2_v1/experiment_2/*/seed_*/*/N1000_balanced_*/results.csv'))
assert len(paths) == 54, f'Expected 54 completed cells, found {len(paths)}. Finish the full-matrix runner first.'
results = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
methods = ['feddf', 'all_teachers_support', 'expert_full', 'expert_support']
method_label = {'feddf': 'FedDF', 'all_teachers_support': 'All + support', 'expert_full': 'EXPERT full', 'expert_support': 'EXPERT + support'}
method_color = {'all_teachers_support': '#e76f51', 'expert_full': '#457b9d', 'expert_support': '#2a9d8f'}
student = results.query("record_type == 'student'")
target = results.query("record_type == 'target'")
coverage = student.groupby('dataset').agg(seeds=('seed', 'nunique'), regimes=('regime', 'nunique'), methods=('method', 'nunique'), cells=('method', 'size'))
display(coverage)
assert set(student.method) == set(methods) and coverage.cells.eq(72).all()

AssertionError: Expected 54 completed cells, found 2. Finish the full-matrix runner first.

In [ ]:
# Primary outcome: paired global-test accuracy change versus FedDF (percentage points).
base = student[student.method.eq('feddf')].set_index(['dataset', 'seed', 'regime']).test_accuracy
effects = student[student.method.ne('feddf')].copy()
effects['delta_pp'] = 100 * (effects.test_accuracy - effects.set_index(['dataset', 'seed', 'regime']).index.map(base))
summary = effects.groupby(['dataset', 'regime', 'method']).delta_pp.agg(['mean', 'sem']).reset_index()
datasets = sorted(summary.dataset.unique()); regimes = ['iid', 'alpha0p1', 'alpha0p5', 'alpha1p0', 'multi', 'single']
fig, axes = plt.subplots(1, len(datasets), figsize=(16, 4), sharey=True, constrained_layout=True)
for ax, dataset in zip(axes, datasets):
    frame = summary[summary.dataset.eq(dataset)].set_index(['regime', 'method'])
    for offset, method in zip([-0.25, 0, 0.25], methods[1:]):
        part = frame.reindex(pd.MultiIndex.from_product([regimes, [method]], names=['regime', 'method'])).reset_index()
        ax.bar([i + offset for i in range(len(regimes))], part['mean'], width=.24, yerr=part['sem'], capsize=2, color=method_color[method], label=method_label[method])
    ax.axhline(0, color='black', lw=.8); ax.set_xticks(range(len(regimes)), regimes, rotation=35, ha='right'); ax.set_title(dataset); ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('Paired change in global accuracy (pp vs FedDF)'); axes[-1].legend(frameon=False, fontsize=8, loc='best')
fig.suptitle('Which mechanism improves the student?', y=1.03); plt.show()
display(summary.pivot(index=['dataset', 'regime'], columns='method', values='mean').round(2))

In [ ]:
# Mechanism visible before training: target true-class mass and entropy, each versus FedDF.
target_base = target[target.method.eq('feddf')].set_index(['dataset', 'seed', 'regime'])
target_effects = target[target.method.ne('feddf')].copy().set_index(['dataset', 'seed', 'regime'])
for metric in ['mean_probability_true_class', 'target_entropy']:
    target_effects[metric + '_delta'] = target_effects[metric] - target_effects.index.map(target_base[metric])
target_effects = target_effects.reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
for ax, metric, title in zip(axes, ['mean_probability_true_class_delta', 'target_entropy_delta'], ['Δ true-class probability', 'Δ target entropy (nats)']):
    grouped = target_effects.groupby(['dataset', 'method'])[metric].mean().unstack('method').reindex(columns=methods[1:])
    grouped.rename(columns=method_label).plot.bar(ax=ax, color=[method_color[m] for m in methods[1:]], width=.75)
    ax.axhline(0, color='black', lw=.8); ax.set_title(title); ax.set_xlabel('Dataset'); ax.grid(axis='y', alpha=.25); ax.legend(frameon=False, fontsize=8)
fig.suptitle('Target transformations before student training', y=1.03); plt.show()